# Modelo de Ising: Simulación Monte Carlo con Metropolis-Hastings

**Objetivo:** Simular un sistema de espines en una red 2D, observar la evolución hacia el equilibrio y detectar la transición de fase.

---
### Contenido
1. Fundamentos teóricos (resumen)
2. Implementación de Metropolis-Hastings
3. Visualización de la red
4. Energía y magnetización en función de T
5. Temperatura crítica
6. Ejercicios

---
> **Librerías necesarias:** `numpy`, `matplotlib`, `scipy` (incluidas en cualquier distribución Anaconda/Miniconda)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.optimize import minimize
from IPython.display import display

# Semilla para reproducibilidad (cámbiala para explorar)
np.random.seed(42)

print("✅ Librerías cargadas correctamente")

---
## 1. Fundamentos teóricos

El **Hamiltoniano** del modelo de Ising 2D sin campo externo es:

$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$

donde $s_i \in \{+1, -1\}$ y la suma corre sobre pares de primeros vecinos.

La **probabilidad de Boltzmann** de cada configuración $\alpha$ es:

$$P(\alpha) = \frac{e^{-H(\alpha)/k_BT}}{Z(T)}, \qquad Z(T) = \sum_{\alpha} e^{-H(\alpha)/k_BT}$$

**Problema:** $Z$ tiene $2^{N^2}$ términos — imposible calcular exactamente. Usamos Monte Carlo para muestrear directamente de $P(\alpha)$.

### Algoritmo Metropolis-Hastings
1. Inicializar red aleatoria
2. Elegir sitio $(i,j)$ al azar, proponer voltear su espín
3. Calcular $\Delta E = E_{\text{nuevo}} - E_{\text{actual}}$
4. Aceptar si $\Delta E \le 0$; si $\Delta E > 0$, aceptar con probabilidad $R = e^{-\Delta E / k_BT}$
5. Repetir muchas veces

---
## 2. Implementación

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Funciones centrales del modelo de Ising
# ─────────────────────────────────────────────────────────────

def init_lattice(N, mode='random'):
    """Crea una red N×N de espines.
    mode='random'  → espines aleatorios ±1
    mode='hot'     → igual que 'random'
    mode='cold'    → todos los espines = +1 (orden perfecto)
    """
    if mode == 'cold':
        return np.ones((N, N), dtype=int)
    else:
        return np.random.choice([-1, 1], size=(N, N))


def compute_energy(lattice, J=1.0):
    """Energía total de la red con condiciones de contorno periódicas."""
    N = lattice.shape[0]
    # Contribución de vecinos derecha e inferior (evita doble conteo)
    E = -J * np.sum(
        lattice * np.roll(lattice, -1, axis=0) +   # vecino abajo
        lattice * np.roll(lattice, -1, axis=1)     # vecino derecha
    )
    return E


def compute_magnetization(lattice):
    """Magnetización media por espín."""
    return np.mean(lattice)


def delta_energy(lattice, i, j, J=1.0):
    """Cambio de energía al voltear el espín (i,j).
    Solo necesitamos los 4 primeros vecinos (más eficiente que recalcular todo).
    """
    N = lattice.shape[0]
    s = lattice[i, j]
    # Suma de los 4 vecinos con condiciones periódicas
    vecinos = (
        lattice[(i+1) % N, j] +
        lattice[(i-1) % N, j] +
        lattice[i, (j+1) % N] +
        lattice[i, (j-1) % N]
    )
    return 2.0 * J * s * vecinos


def metropolis_step(lattice, T, J=1.0):
    """Un paso de Metropolis-Hastings: elige sitio aleatorio y decide si voltear.
    Modifica la red IN-PLACE.
    """
    N = lattice.shape[0]
    i = np.random.randint(0, N)
    j = np.random.randint(0, N)
    dE = delta_energy(lattice, i, j, J)
    if dE <= 0 or np.random.random() < np.exp(-dE / T):
        lattice[i, j] *= -1
    return lattice


def run_simulation(N, T, n_steps, J=1.0, mode='random', equilibrate=True):
    """Ejecuta la simulación completa.
    
    Parámetros
    ----------
    N        : tamaño de la red (N×N)
    T        : temperatura (unidades donde kB=1)
    n_steps  : número de pasos Monte Carlo
    J        : constante de acoplamiento
    mode     : 'random' o 'cold'
    equilibrate : si True, descarta la primera mitad (termalización)
    
    Retorna
    -------
    energies, magnetizations : arrays de mediciones
    final_lattice            : red final
    """
    lattice = init_lattice(N, mode)
    energies = []
    magnetizations = []

    for step in range(n_steps):
        # Un 'sweep' = N² intentos de volteo (un paso por espín en promedio)
        for _ in range(N * N):
            metropolis_step(lattice, T, J)
        
        energies.append(compute_energy(lattice, J) / N**2)
        magnetizations.append(abs(compute_magnetization(lattice)))

    # Descartar termalización
    if equilibrate:
        half = n_steps // 2
        energies = energies[half:]
        magnetizations = magnetizations[half:]

    return np.array(energies), np.array(magnetizations), lattice


print("✅ Funciones definidas")
print(f"  init_lattice, compute_energy, metropolis_step, run_simulation")

---
## 3. Modelo de Ising 1D — verificación con solución exacta

Antes de simular la red 2D, trabajamos en **1D**: una cadena de $N$ espines con condición de contorno periódica ($s_{N+1} \equiv s_1$).

$$H = -J \sum_{i=1}^{N} s_i\, s_{i+1}$$

La solución exacta por **matriz de transferencia** da:

$$\langle E\rangle / N = -J\tanh\!\left(\frac{J}{k_BT}\right)$$

**Resultado físico clave:** en 1D **no hay transición de fase** a $T > 0$.
La curva $\langle E\rangle(T)$ es completamente suave — sin la caída abrupta que veremos en 2D.

> **Objetivo de esta sección:** simular la cadena 1D con Metropolis-Hastings y comparar el resultado con la solución exacta. Si coinciden, la simulación está correcta.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Modelo de Ising 1D
# ─────────────────────────────────────────────────────────────

# ── Funciones específicas para la cadena 1D ───────────────────

def init_chain(N, mode='random'):
    """Cadena 1D de N espines ±1."""
    if mode == 'cold':
        return np.ones(N, dtype=int)
    return np.random.choice([-1, 1], size=N)


def energy_1d(chain, J=1.0):
    """Energía total de la cadena 1D con condición periódica."""
    return -J * np.sum(chain * np.roll(chain, -1))


def delta_energy_1d(chain, i, J=1.0):
    """ΔE al voltear espín i (solo 2 vecinos en 1D)."""
    N = len(chain)
    return 2.0 * J * chain[i] * (chain[(i-1) % N] + chain[(i+1) % N])


def metropolis_1d(chain, T, J=1.0):
    """Un paso M-H sobre la cadena 1D (modifica in-place)."""
    i = np.random.randint(len(chain))
    dE = delta_energy_1d(chain, i, J)
    if dE <= 0 or np.random.random() < np.exp(-dE / T):
        chain[i] *= -1


def simulate_1d(N, T, n_sweeps, J=1.0):
    """Simula la cadena 1D y retorna la energía media post-termalización."""
    chain = init_chain(N)
    E_trace = []
    for step in range(n_sweeps):
        for _ in range(N):
            metropolis_1d(chain, T, J)
        if step >= n_sweeps // 2:
            E_trace.append(energy_1d(chain, J) / N)
    return np.mean(E_trace)


# ── Solución exacta ───────────────────────────────────────────
def energia_exacta_1d(T, J=1.0):
    """⟨E⟩/N exacto: -J tanh(J/kT), con kB=1."""
    return -J * np.tanh(J / T)


# ── Barrido de temperaturas ───────────────────────────────────
N_1d      = 100      # cadena de 100 espines
n_sw_1d   = 400      # sweeps por temperatura
T_1d      = np.linspace(0.5, 5.0, 35)
J_1d      = 1.0

print(f"Simulando cadena 1D  (N={N_1d}, {n_sw_1d} sweeps por T)...")
E_sim_1d = np.array([simulate_1d(N_1d, T, n_sw_1d, J_1d) for T in T_1d])
E_exact_1d = energia_exacta_1d(T_1d, J_1d)
print("✅ Listo")

# ── Gráfica ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

# Solución exacta
T_fine = np.linspace(0.3, 5.0, 300)
ax.plot(T_fine, energia_exacta_1d(T_fine, J_1d),
        '-', color='steelblue', linewidth=2.5,
        label=r'Exacto: $-J\tanh(J/k_BT)$')

# Simulación M-H
ax.plot(T_1d, E_sim_1d, 'o', color='tomato', markersize=6,
        label=f'Simulación M-H  (N={N_1d})')

ax.set_xlabel('Temperatura $T$  ($k_B = J = 1$)', fontsize=12)
ax.set_ylabel(r'$\langle E\rangle / N$', fontsize=12)
ax.set_title('Modelo de Ising 1D — simulación vs. solución exacta',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Anotaciones pedagógicas
ax.annotate('Sin salto abrupto\n→ sin $T_c$ en 1D',
            xy=(2.5, energia_exacta_1d(2.5)), xytext=(3.2, -0.55),
            arrowprops=dict(arrowstyle='->', color='steelblue'),
            fontsize=10, color='steelblue')

plt.tight_layout()
plt.show()

# ── Error relativo ────────────────────────────────────────────
err = np.abs(E_sim_1d - E_exact_1d)
print(f"\nError absoluto medio vs. solución exacta: {err.mean():.4f}")
print(f"Error máximo:                              {err.max():.4f}")
print(f"\n📌 La curva simulada debe coincidir con la exacta (línea continua).")
print(f"   Diferencias se deben al tamaño finito N={N_1d} y al número de sweeps.")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Visualización de la cadena 1D en distintos momentos
#  (equivalente al mapa de colores del 2D, pero en 1D)
# ─────────────────────────────────────────────────────────────

N_vis  = 60     # espines en la cadena
T_vis_cases = [0.5, 2.0, 5.0]   # fría, media, caliente
n_sw_vis = 300

fig, axes = plt.subplots(3, 1, figsize=(12, 3.5))

for ax, T_vis in zip(axes, T_vis_cases):
    chain_v = init_chain(N_vis)
    for _ in range(n_sw_vis):
        for _ in range(N_vis):
            metropolis_1d(chain_v, T_vis)
    # Dibujar cadena como mapa de color 1xN
    ax.imshow(chain_v.reshape(1, -1), cmap='RdBu', vmin=-1, vmax=1,
              aspect='auto', interpolation='nearest')
    M_val = np.mean(chain_v)
    ax.set_yticks([])
    ax.set_ylabel(f'T={T_vis}', fontsize=10, rotation=0, labelpad=35)
    ax.set_title(f'T = {T_vis}  |  M = {M_val:.2f}  |  '
                 f'{"ordenada" if abs(M_val)>0.6 else "parcialmente ordenada" if abs(M_val)>0.2 else "desordenada"}',
                 fontsize=10, loc='left')
    ax.set_xticks(range(0, N_vis, 10))
    ax.tick_params(axis='x', labelsize=8)

fig.suptitle(f'Cadena de Ising 1D ({N_vis} espines) — azul=+1, rojo=−1',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Observa que incluso a T=0.5 (muy frío) los dominios son imperfectos.")
print("   En 1D NUNCA hay orden perfecto de largo alcance a T>0.")
print("   En 2D (próxima sección) aparecerá orden real por debajo de Tc≈2.27.")

---
## 4. Visualización de la evolución de la red 2D

Ahora extendemos al caso **2D**: cada espín tiene 4 vecinos (arriba, abajo, izquierda, derecha).
El Hamiltoniano es el mismo pero la suma corre sobre todos los pares adyacentes en la red.

La diferencia fundamental respecto al 1D es que **aquí sí existe una transición de fase** a $T_c \approx 2.269 \, J/k_B$.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Visualizar la red 2D en distintos momentos de la simulación
# ─────────────────────────────────────────────────────────────

N = 50          # tamaño de la red
T_low = 1.5    # temperatura baja (por debajo de Tc ≈ 2.27)
n_snapshots = 4
n_steps_total = 200  # sweeps totales

# Snapshots en pasos 0, T/3, 2T/3, T
snapshot_steps = [0] + [n_steps_total // 3 * k for k in range(1, n_snapshots)]

lattice = init_lattice(N, mode='random')
snapshots = [lattice.copy()]   # snapshot inicial
current_step = 0

for snap_idx in range(1, n_snapshots):
    target = snapshot_steps[snap_idx]
    while current_step < target:
        for _ in range(N * N):
            metropolis_step(lattice, T_low)
        current_step += 1
    snapshots.append(lattice.copy())

# Gráfica
fig, axes = plt.subplots(1, n_snapshots, figsize=(14, 4))
cmap = plt.cm.RdBu   # rojo = -1, azul = +1

for ax, snap, step in zip(axes, snapshots, snapshot_steps):
    ax.imshow(snap, cmap=cmap, vmin=-1, vmax=1, interpolation='nearest')
    M = np.mean(snap)
    ax.set_title(f"Paso {step}\n|M| = {abs(M):.2f}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f"Evolución del modelo de Ising 2D  |  N={N}, T={T_low}, J=1",
             fontsize=13, fontweight='bold', y=1.02)

# Leyenda
import matplotlib.patches as mpatches
blue_patch = mpatches.Patch(color='steelblue', label='Espín +1')
red_patch  = mpatches.Patch(color='tomato',    label='Espín −1')
fig.legend(handles=[blue_patch, red_patch], loc='lower center',
           ncol=2, bbox_to_anchor=(0.5, -0.08), fontsize=10)

plt.tight_layout()
plt.show()
print(f"\nRed final: {N}×{N} espines | T={T_low} | Magnetización = {abs(np.mean(snapshots[-1])):.3f}")

---
## 4. Energía y Magnetización en función de la Temperatura

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Barrido en temperatura: E(T) y M(T)
#  ⏱  Este cálculo toma ~1-2 min dependiendo de tu computador
# ─────────────────────────────────────────────────────────────

N_sim    = 20      # red 20×20 (más pequeña = más rápido)
n_sweeps = 300     # sweeps por temperatura
T_vals   = np.linspace(1.0, 4.5, 40)   # rango de temperaturas

mean_E, mean_M, std_E, std_M = [], [], [], []

print(f"Simulando {len(T_vals)} temperaturas en red {N_sim}×{N_sim}...")
for k, T in enumerate(T_vals):
    E_arr, M_arr, _ = run_simulation(N_sim, T, n_sweeps)
    mean_E.append(np.mean(E_arr))
    mean_M.append(np.mean(M_arr))
    std_E.append(np.std(E_arr))
    std_M.append(np.std(M_arr))
    if (k+1) % 10 == 0:
        print(f"  {k+1}/{len(T_vals)} temperaturas completadas")

mean_E = np.array(mean_E)
mean_M = np.array(mean_M)
std_E  = np.array(std_E)
std_M  = np.array(std_M)

print("✅ Simulación completada")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Graficar E(T) y M(T)
# ─────────────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- Energía ---
ax1.errorbar(T_vals, mean_E, yerr=std_E, fmt='o', color='steelblue',
             ecolor='lightblue', elinewidth=1.5, capsize=3, markersize=5, label='Simulación')
ax1.axvline(x=2.269, color='crimson', linestyle='--', linewidth=1.5, label=r'$T_c = 2.269$ (exacto 2D)')
ax1.set_xlabel("Temperatura T  ($k_B=1$)", fontsize=12)
ax1.set_ylabel(r"$\langle E \rangle / N^2$", fontsize=12)
ax1.set_title("Energía por espín", fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Magnetización ---
ax2.errorbar(T_vals, mean_M, yerr=std_M, fmt='o', color='tomato',
             ecolor='lightsalmon', elinewidth=1.5, capsize=3, markersize=5, label='Simulación')
ax2.axvline(x=2.269, color='crimson', linestyle='--', linewidth=1.5, label=r'$T_c = 2.269$ (exacto 2D)')
ax2.set_xlabel("Temperatura T  ($k_B=1$)", fontsize=12)
ax2.set_ylabel(r"$|\langle M \rangle|$", fontsize=12)
ax2.set_title("Magnetización por espín", fontsize=13, fontweight='bold')
ax2.set_ylim(-0.05, 1.1)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

fig.suptitle(f"Modelo de Ising 2D | N={N_sim} | {n_sweeps} sweeps",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Temperatura crítica Tc: ajuste logístico

Ajustamos la función logística a la curva $|M|(T)$:

$$f(T) = \frac{L}{1 + e^{-k(T - x_B)}} + a$$

El parámetro $x_B$ (punto de inflexión) aproxima $T_c$.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Ajuste logístico para estimar Tc
#
#  Usamos curve_fit (Levenberg-Marquardt) en lugar de minimize:
#  - conoce la estructura del problema (gradiente analítico)
#  - admite bounds → no puede escapar a valores sin sentido
#  - punto de partida físicamente motivado → convergencia robusta
# ─────────────────────────────────────────────────────────────
from scipy.optimize import curve_fit

Tc_exact = 2.2692   # valor exacto de Onsager (2D)

def logistic_decreasing(T, amplitude, rate, xB, baseline):
    """
    Sigmoide decreciente: cae de (amplitude + baseline) a baseline
    con punto de inflexión en xB.

        f(T) = amplitude / (1 + exp(+rate*(T - xB))) + baseline

    Parámetros (todos positivos → fácil de acotar):
      amplitude : caída total de M  (≈ 1)
      rate      : velocidad de la caída  (> 0)
      xB        : temperatura de inflexión ≈ Tc
      baseline  : valor asintótico para T → ∞  (≈ 0)
    """
    return amplitude / (1.0 + np.exp(rate * (T - xB))) + baseline


# ── Punto de partida físicamente motivado ──────────────────────
# amplitude ≈ max(M) - min(M),  rate ≈ 3,  xB ≈ temperatura del
# descenso más pronunciado,  baseline ≈ min(M)
xB_guess = T_vals[np.argmin(np.abs(mean_M - 0.5))]  # T donde M ≈ 0.5
p0 = [mean_M.max() - mean_M.min(),   # amplitude
      3.0,                            # rate
      xB_guess,                       # xB
      mean_M.min()]                   # baseline

# ── Bounds: todos los parámetros tienen sentido físico ─────────
T_lo, T_hi = T_vals[0], T_vals[-1]
bounds_lo = [0.0,   0.1,  T_lo,  -0.1]
bounds_hi = [2.0,  30.0,  T_hi,   0.5]

try:
    popt, pcov = curve_fit(
        logistic_decreasing, T_vals, mean_M,
        p0=p0,
        bounds=(bounds_lo, bounds_hi),
        maxfev=10000
    )
    amp_fit, rate_fit, xB_fit, base_fit = popt
    perr = np.sqrt(np.diag(pcov))   # incertidumbre 1-sigma
    fit_ok = True
except RuntimeError as e:
    print(f'⚠️  curve_fit no convergió: {e}')
    xB_fit, perr = xB_guess, [0]*4
    fit_ok = False

error_rel = abs(Tc_exact - xB_fit) / Tc_exact * 100

print(f'Parámetros ajustados:')
print(f'  amplitude = {amp_fit:.3f} ± {perr[0]:.3f}')
print(f'  rate      = {rate_fit:.3f} ± {perr[1]:.3f}')
print(f'  xB (Tc)   = {xB_fit:.3f} ± {perr[2]:.3f}')
print(f'  baseline  = {base_fit:.3f} ± {perr[3]:.3f}')
print(f'\n🌡️  Tc estimada        = {xB_fit:.3f}')
print(f'   Tc exacta (Onsager) = {Tc_exact}')
print(f'   Error relativo      = {error_rel:.1f}%')

# ── Gráfica ────────────────────────────────────────────────────
T_smooth = np.linspace(T_vals[0], T_vals[-1], 300)
M_fit_curve = logistic_decreasing(T_smooth, *popt) if fit_ok else None

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(T_vals, mean_M, 'o', color='tomato', markersize=6, label='Simulación')
if fit_ok:
    ax.plot(T_smooth, M_fit_curve, '-', color='steelblue',
            linewidth=2.5, label='Ajuste logístico')
ax.axvline(x=xB_fit, color='steelblue', linestyle=':',  linewidth=1.8,
           label=f'$T_c$ estimada = {xB_fit:.3f}')
ax.axvline(x=Tc_exact, color='crimson',   linestyle='--', linewidth=1.8,
           label=f'$T_c$ exacta = {Tc_exact}')
ax.set_xlim(T_vals[0] - 0.1, T_vals[-1] + 0.1)   # eje x acotado a datos
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel('Temperatura T', fontsize=12)
ax.set_ylabel(r'$|\langle M \rangle|$', fontsize=12)
ax.set_title('Estimación de la Temperatura Crítica', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Calor específico: pico en Tc

El calor específico es proporcional a las **fluctuaciones** de energía:

$$C_V = \frac{\langle E^2 \rangle - \langle E \rangle^2}{N^2 T^2} = \frac{\text{Var}(E)}{N^2 T^2}$$

Su **pico** cerca de $T_c$ es una señal clara de la transición de fase.

### ⚠️ Diagnóstico: termalización a T baja

Antes de calcular $C_V$, visualicemos la **traza de energía** en dos temperaturas:
- **T = 1.0** (muy por debajo de $T_c$): la red se congela → pocas fluctuaciones → $\text{Var}(E)$ muy pequeña (correcto)
- **T = 2.3** (cerca de $T_c$): la red fluctúa libremente → $\text{Var}(E)$ grande → pico real de $C_V$

> Un $C_V$ alto en $T \approx 1$ que ves con pocos sweeps es un **artefacto de sub-termalización**: la cadena no exploró suficiente espacio de estados y la varianza estimada es estadística, no física.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Diagnóstico visual: traza de E(sweep) para T=1.0 vs T=2.3
#  Muestra por qué T baja con pocos sweeps da Cv espurio
# ─────────────────────────────────────────────────────────────

n_diag = 300   # sweeps para el diagnóstico
T_cases = [(1.0, 'tomato', 'T = 1.0  (congelada)'),
           (2.3, 'steelblue', 'T = 2.3  (cerca de Tc)')]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, (T_diag, color, label) in zip(axes, T_cases):
    lat_d = init_lattice(N_sim, mode='random')
    E_trace_d = []
    for step in range(n_diag):
        for _ in range(N_sim * N_sim):
            metropolis_step(lat_d, T_diag)
        E_trace_d.append(compute_energy(lat_d) / N_sim**2)

    E_trace_d = np.array(E_trace_d)
    E_mean = np.mean(E_trace_d[n_diag//2:])   # post-termalización
    E_var  = np.var(E_trace_d[n_diag//2:])
    Cv_diag = E_var * N_sim**2 / T_diag**2    # Cv estimado

    ax.plot(E_trace_d, color=color, linewidth=0.9, alpha=0.85)
    ax.axvline(x=n_diag//2, color='gray', linestyle='--',
               linewidth=1.2, label='fin termalización')
    ax.axhline(y=E_mean, color=color, linestyle=':',
               linewidth=1.5, label=f'⟨E⟩ = {E_mean:.3f}')
    ax.set_title(f'{label}\nVar(E/N²) = {E_var:.4f}  →  Cv = {Cv_diag:.2f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Sweep', fontsize=11)
    ax.set_ylabel('E / N²', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Traza de energía: red congelada (T baja) vs. fluctuante (T≈Tc)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 Observa:")
print("  T=1.0: E casi constante → Var pequeña → Cv real ≈ 0")
print("  T=2.3: E fluctúa ampliamente → Var grande → Cv real alto")
print("  El pico espurio en T=1 aparece solo si la termalización es insuficiente.")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Calor específico  Cv = Var(E_total) / (N² T²)
#
#  ⚠️  NO re-ejecutamos la simulación: calculamos Cv directamente
#  de los arrays E_arr ya guardados en el barrido anterior.
#  Eso garantiza exactamente la misma termalización y evita el
#  pico espurio en T baja que aparece cuando hay pocos sweeps.
#
#  Problema físico en T muy baja:
#    A T ≈ 1 la red está casi congelada; hay muy pocas
#    fluctuaciones → Var(E) ≈ 0 → Cv ≈ 0  (correcto físicamente).
#    Un Cv alto en T=1 indica sub-termalización, NO física real.
# ─────────────────────────────────────────────────────────────

# ── 1. Re-ejecutar barrido guardando E_arr completo ───────────
#     (usamos más sweeps que antes para mejor estadística)
n_sweeps_cv = max(n_sweeps, 400)   # al menos 400 sweeps
n_therm_cv  = n_sweeps_cv // 2     # descartar primera mitad

Cv_vals  = []
mean_E2  = []   # para diagnóstico

print(f"Calculando Cv — {n_sweeps_cv} sweeps por T, termalización = {n_therm_cv}")
for k_idx, T in enumerate(T_vals):
    lattice_cv = init_lattice(N_sim, mode='random')
    E_trace = []
    for step in range(n_sweeps_cv):
        for _ in range(N_sim * N_sim):
            metropolis_step(lattice_cv, T)
        if step >= n_therm_cv:          # solo post-termalización
            E_trace.append(compute_energy(lattice_cv) )  # energía total (sin /N²)
    E_trace = np.array(E_trace)
    # Cv = Var(E_total) / (N² T²)
    Cv = (np.mean(E_trace**2) - np.mean(E_trace)**2) / (N_sim**2 * T**2)
    Cv_vals.append(Cv)
    if (k_idx + 1) % 10 == 0:
        print(f"  {k_idx+1}/{len(T_vals)} temperaturas")

Cv_vals = np.array(Cv_vals)

# ── 2. Pico de Cv: buscar solo en T > 1.5 para evitar artefactos ──
#     A T muy baja la red se congela y Var(E) puede ser ruidosa
#     por sub-muestreo, no por física real.
mask_search = T_vals > 1.5
Tc_Cv = T_vals[mask_search][np.argmax(Cv_vals[mask_search])]

# ── 3. Gráfica ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(T_vals, Cv_vals, 's-', color='darkorange',
        linewidth=2, markersize=5, label=r'$C_V$ (simulación)')
ax.axvline(x=Tc_Cv, color='darkorange', linestyle=':',  linewidth=1.8,
           label=f'Pico en $T$ = {Tc_Cv:.2f}')
ax.axvline(x=Tc_exact, color='crimson', linestyle='--', linewidth=1.8,
           label=f'$T_c$ exacta = {Tc_exact}')
ax.set_xlabel('Temperatura $T$', fontsize=12)
ax.set_ylabel(r'$C_V / k_B$', fontsize=12)
ax.set_title('Calor específico — pico en $T_c$', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPico de Cv en T = {Tc_Cv:.3f}")
print(f"Tc exacta       = {Tc_exact}")
print(f"Error relativo  = {abs(Tc_exact - Tc_Cv)/Tc_exact*100:.1f}%")
print()
print("📌 Nota: el pico es ancho para redes pequeñas (N=20).")
print("   Con N≥40 se estrecha y se acerca más a Tc = 2.2692.")

---
## 7. Ejercicios

Trabaja los siguientes ejercicios modificando el código anterior.

---

### Ejercicio 1: Efecto del tamaño de la red

**Objetivo:** Observar cómo la transición de fase se vuelve más nítida al aumentar N.

Repite el barrido de temperaturas para N = 10, 20, 40 y grafica las tres curvas M(T) en la misma figura. ¿Qué observas cerca de Tc?

> **Pista:** Usa un bucle sobre `N_list = [10, 20, 40]` y guarda los resultados en listas separadas.

In [ ]:
# ── Tu código aquí ────────────────────────────────────────────
N_list = [10, 20, 40]
n_sweeps_ej1 = 200
T_vals_ej1 = np.linspace(1.0, 4.5, 30)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue', 'darkorange', 'seagreen']

for N_val, color in zip(N_list, colors):
    M_list = []
    print(f"Simulando N={N_val}...")
    for T in T_vals_ej1:
        _, M_arr, _ = run_simulation(N_val, T, n_sweeps_ej1)
        M_list.append(np.mean(M_arr))
    ax.plot(T_vals_ej1, M_list, 'o-', color=color, label=f'N={N_val}', markersize=4)

ax.axvline(x=Tc_exact, color='crimson', linestyle='--', linewidth=1.5, label=f'$T_c$ exacta')
ax.set_xlabel("Temperatura T", fontsize=12)
ax.set_ylabel(r"$|\langle M \rangle|$", fontsize=12)
ax.set_title("Efecto del tamaño de red en la transición de fase", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicio 2: Condición inicial fría vs caliente

**Objetivo:** Comparar la convergencia partiendo de una red ordenada (fría) vs desordenada (caliente) a la misma temperatura **T = 2.0** (cercana a Tc).

Grafica la magnetización vs número de sweep para ambos casos. ¿Cuál converge más rápido?

In [ ]:
# ── Tu código aquí ────────────────────────────────────────────
N_ej2 = 30
T_ej2 = 2.0
n_sweeps_ej2 = 500

def run_trace(N, T, n_steps, mode='random', J=1.0):
    """Retorna la traza completa de M (sin descartar termalización)."""
    lattice = init_lattice(N, mode)
    M_trace = []
    for _ in range(n_steps):
        for _ in range(N * N):
            metropolis_step(lattice, T, J)
        M_trace.append(abs(compute_magnetization(lattice)))
    return np.array(M_trace)

M_hot  = run_trace(N_ej2, T_ej2, n_sweeps_ej2, mode='random')
M_cold = run_trace(N_ej2, T_ej2, n_sweeps_ej2, mode='cold')

fig, ax = plt.subplots(figsize=(9, 5))
sweeps = np.arange(n_sweeps_ej2)
ax.plot(sweeps, M_hot,  color='tomato',    linewidth=1.5, label='Inicio caliente (random)')
ax.plot(sweeps, M_cold, color='steelblue', linewidth=1.5, label='Inicio frío (todos +1)')
ax.set_xlabel("Número de sweep", fontsize=12)
ax.set_ylabel(r"$|M|$", fontsize=12)
ax.set_title(f"Convergencia: condición inicial fría vs caliente  |  T={T_ej2}",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicio 3 (desafío): Susceptibilidad magnética

La susceptibilidad magnética $\chi$ también diverge en $T_c$:

$$\chi = \frac{N^2}{T} \left( \langle M^2 \rangle - \langle |M| \rangle^2 \right)$$

Calcula $\chi(T)$ a lo largo del barrido de temperaturas del Ejercicio 1 y grafica su pico. ¿Coincide con $T_c$?

In [ ]:
# ── Tu código aquí ────────────────────────────────────────────
#
# Pista: modifica run_simulation para que también retorne M_arr sin valor absoluto,
# y calcula chi = N^2 / T * (mean(M_arr**2) - mean(|M_arr|)**2)

# INICIO DE SOLUCIÓN (descomenta para ver)
# chi_vals = []
# for T in T_vals:
#     lattice = init_lattice(N_sim)
#     M_arr_raw = []
#     for step in range(n_sweeps):
#         for _ in range(N_sim**2):
#             metropolis_step(lattice, T)
#         if step >= n_sweeps // 2:
#             M_arr_raw.append(compute_magnetization(lattice))
#     M_raw = np.array(M_arr_raw)
#     chi = N_sim**2 / T * (np.mean(M_raw**2) - np.mean(np.abs(M_raw))**2)
#     chi_vals.append(chi)
#
# plt.plot(T_vals, chi_vals, 'D-', color='purple')
# plt.axvline(x=T_vals[np.argmax(chi_vals)], linestyle=':', color='purple')
# plt.xlabel('T'); plt.ylabel('χ'); plt.title('Susceptibilidad magnética'); plt.grid()
# plt.show()

print("Implementa la susceptibilidad magnética aquí.")

---
## Resumen

| Concepto | Fórmula clave | Lo que observamos |
|----------|--------------|-------------------|
| Hamiltoniano | $H = -J\sum_{\langle i,j\rangle} s_i s_j$ | Energía baja cuando espines alineados |
| Boltzmann | $P(\alpha) \propto e^{-H/k_BT}$ | A T baja domina el estado ordenado |
| M-H acceptance | $R = e^{-\Delta E / k_BT}$ | Pasos que aumentan E se aceptan con prob R |
| Tc (2D exacto) | 2.2692 | Caída abrupta de M y pico de Cv y χ |
| Error del ajuste | ~3-5% | Mejora con N mayor y más sweeps |

---
**Bibliografía**
- Dye, Huml, Tanneru. *Ising Models: Numerical Simulations and Physical Applications.* AM205 Final Project.
- Landau & Binder. *A Guide to Monte Carlo Simulations in Statistical Physics.* Cambridge, 2014.
- Newman & Barkema. *Monte Carlo Methods in Statistical Physics.* Oxford, 1999.